# TabICLv2 Regressor artifact inference — standalone Colab

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/tabicl-regressor-pipeline)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/tabicl-regressor-pipeline/blob/main/tutorials/tabiclv2_regressor_artifact_inference_colab.ipynb)

Load a portable artifact produced by the main tutorial or a compatible DIMER serving bundle. This notebook performs **no gradient fine-tuning**. TabICL's required `fit(context_X, context_y)` call only registers the in-context support table before prediction.

> **Trust boundary:** load only artifacts you created yourself or obtained from a trusted source. Safe ZIP extraction prevents path traversal; it does not make a PyTorch checkpoint trustworthy. Paste a known ZIP SHA-256 below when available.


## 1. Install


In [ ]:
%pip -q install "tabicl==2.1.1" "pyarrow>=15" "pandas>=2" "scikit-learn>=1.4"

import importlib.metadata, torch
TABICL_VERSION="2.1.1"
if importlib.metadata.version("tabicl") != TABICL_VERSION: raise RuntimeError("Unexpected tabicl version")
print("TabICL:",TABICL_VERSION,"CUDA:",torch.cuda.is_available())


## 2. Upload and verify the artifact ZIP


In [ ]:
from pathlib import Path
from google.colab import files
import csv, io, hashlib, json, shutil, stat, zipfile
import numpy as np, pandas as pd

EXPECTED_ZIP_SHA256=""  # @param {type:"string"}

def sha256_file(path):
    h=hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda:f.read(1<<20),b""): h.update(chunk)
    return h.hexdigest()

def safe_extract_zip(zip_path,dest):
    dest=Path(dest); dest.mkdir(parents=True,exist_ok=True); root=dest.resolve()
    with zipfile.ZipFile(zip_path) as z:
        for info in z.infolist():
            name=info.filename.replace("\\","/"); parts=Path(name).parts; mode=info.external_attr>>16
            if name.startswith("/") or ".." in parts or stat.S_ISLNK(mode): raise ValueError(f"Unsafe ZIP member: {info.filename}")
            out=(dest/Path(name)).resolve()
            if root != out and root not in out.parents: raise ValueError("ZIP member escapes destination")
        z.extractall(dest)

uploaded=files.upload()
if len(uploaded)!=1: raise ValueError("Upload exactly one artifact ZIP")
name,payload=next(iter(uploaded.items()))
zip_path=Path("/content")/Path(name).name; zip_path.write_bytes(payload)
observed=sha256_file(zip_path); print("ZIP SHA-256:",observed)
if EXPECTED_ZIP_SHA256:
    expected=EXPECTED_ZIP_SHA256.strip().lower()
    if len(expected)!=64 or any(c not in "0123456789abcdef" for c in expected): raise ValueError("EXPECTED_ZIP_SHA256 must be 64 hex chars")
    if observed!=expected: raise RuntimeError("Artifact ZIP SHA-256 mismatch")
extract_dir=Path("/content/tabiclv2-regressor-artifact")
if extract_dir.exists(): shutil.rmtree(extract_dir)
safe_extract_zip(zip_path,extract_dir)
matches=list(extract_dir.rglob("artifact.json"))
if len(matches)!=1: raise ValueError("Expected exactly one artifact.json")
root=matches[0].parent; manifest=json.loads(matches[0].read_text())
if manifest.get("artifactFormat")!="tabicl-dimer-regressor-v1": raise ValueError(f"Unsupported artifactFormat: {manifest.get('artifactFormat')}")
if manifest.get("tabiclVersion")!=TABICL_VERSION: raise ValueError("Artifact TabICL version does not match notebook pin")
def manifest_member_path(root, value, field):
    rel=Path(str(value))
    if rel.is_absolute() or ".." in rel.parts:
        raise ValueError(f"Unsafe {field} path in artifact.json: {value!r}")
    root_resolved=Path(root).resolve(); out=(root_resolved/rel).resolve()
    if root_resolved != out and root_resolved not in out.parents:
        raise ValueError(f"{field} path escapes artifact root: {value!r}")
    return out
ckpt=manifest_member_path(root,manifest["checkpoint"],"checkpoint")
context_path=manifest_member_path(root,manifest["trainingContext"],"trainingContext")
if sha256_file(ckpt)!=manifest["digests"]["checkpointSha256"]: raise RuntimeError("Checkpoint digest mismatch")
if sha256_file(context_path)!=manifest["digests"]["trainingContextSha256"]: raise RuntimeError("Training-context digest mismatch")
print("✓ Artifact structure and digests verified")


## 3. Reconstruct the in-context regressor


In [ ]:
from tabicl import TabICLRegressor
DEVICE="cuda" if torch.cuda.is_available() else "cpu"
context=pd.read_parquet(context_path)
FEATURE_COLUMNS=manifest["featureColumns"]; TARGET_COLUMN=manifest["targetColumn"]; inference=manifest["inference"]
model=TabICLRegressor(model_path=str(ckpt),allow_auto_download=False,n_estimators=inference["nEstimators"],random_state=inference["randomState"],device=DEVICE)
model.fit(context[FEATURE_COLUMNS],context[TARGET_COLUMN])
print(f"✓ Loaded with {len(context)} context rows and {len(FEATURE_COLUMNS)} features")


## 4. Upload rows and predict


In [ ]:
def raw_header(payload):
    reader=csv.reader(io.StringIO(payload.decode("utf-8-sig")))
    for row in reader:
        if row and any(x.strip() for x in row): return row
    raise ValueError("CSV has no header")

def read_inference_csv(payload,feature_columns):
    header=raw_header(payload); seen=set(); dupes=[]
    for name in header:
        if name in seen and name not in dupes: dupes.append(name)
        seen.add(name)
    if dupes: raise ValueError(f"Inference CSV contains duplicate column names: {dupes}")
    frame=pd.read_csv(io.BytesIO(payload))
    if "prediction" in frame.columns: raise ValueError("Inference CSV already contains a 'prediction' column")
    missing=[c for c in feature_columns if c not in frame.columns]
    if missing: raise ValueError(f"Inference CSV missing features: {missing}")
    return frame

def apply_encoder(frame,encoders):
    out=frame.copy()
    for col,cats in encoders.items():
        lookup,unknown={c:i for i,c in enumerate(cats)},len(cats)
        out[col]=[unknown if pd.isna(v) else lookup.get(str(v),unknown) for v in out[col]]
    return out

uploaded=files.upload()
if len(uploaded)!=1: raise ValueError("Upload exactly one inference CSV")
_,payload=next(iter(uploaded.items()))
rows=read_inference_csv(payload,FEATURE_COLUMNS)
X=apply_encoder(rows[FEATURE_COLUMNS],inference.get("categoricalEncoders",{}))
out=rows.copy(); out["prediction"]=np.asarray(model.predict(X),dtype=float)
output_path=Path("/content/tabiclv2_regressor_predictions.csv")
out.to_csv(output_path,index=False); print("✓",output_path); files.download(str(output_path))


## AI provenance

This inference tutorial was developed with substantial AI assistance using **GPT-5.6 Sol High**, via **OpenAI / ChatGPT**, under Agent Relay role **Builder**, with maintainer direction and review. Provenance only; not independent sign-off.
